# Mistral OCR-4 — Nvidia 10-Q Form Analysis

This notebook applies Mistral OCR-4 to the **Nvidia 10-Q SEC Filing** and demonstrates five table-handling and output capabilities:

| # | Feature | Parameter |
|---|---------|---|
| 1 | **Markdown Output** | Default response `.markdown` field |
| 2 | **Page-6 Table Handling** | Fifth option: normalize empty cells before DataFrame conversion |
| 3 | **Paragraph-Level Bounding Boxes** | `include_blocks: true` — `text` / `title` / `list` block types |
| 4 | **Regional-Level Confidence** | `confidence_scores_granularity: 'page'` |
| 5 | **Bounding Box Classification** | `include_blocks: true` — all 13 semantic block types |

> **Document**: `samples/Nvidia-10-Q-Form.pdf` (pages 1–30 of 45; API limit is 30 pages)  
> **Model**: `mistral-ocr-4-0` via Azure AI Foundry

---
## 0. Setup

In [ ]:
import base64
import json
import os
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import warnings
from collections import Counter
from IPython.display import Markdown, display
from dotenv import load_dotenv
from typing import Dict, Any, List

warnings.filterwarnings('ignore')
print('Libraries loaded successfully')

In [ ]:
load_dotenv()

ENDPOINT   = os.getenv('AZURE_MISTRAL_DOCUMENT_AI_ENDPOINT')
API_KEY    = os.getenv('AZURE_MISTRAL_DOCUMENT_AI_KEY')
MODEL_NAME = os.getenv('AZURE_AI_DEPLOYMENT_NAME')

HEADERS = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_KEY}',
}

SAMPLES_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'samples')
# API limit is 30 pages; use the pre-split version (pages 1-30 of the 45-page original)
NVIDIA_PDF  = os.path.join(SAMPLES_DIR, 'Nvidia-10-Q-Form-p1-30.pdf')

print(f'Endpoint : {ENDPOINT}')
print(f'Model    : {MODEL_NAME}')
print(f'PDF path : {NVIDIA_PDF}')
print(f'PDF size : {os.path.getsize(NVIDIA_PDF):,} bytes')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def encode_file(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


def ocr_request(payload: dict) -> dict:
    resp = requests.post(url=ENDPOINT, json=payload, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


# OCR-4 semantic block type → display color
BLOCK_COLORS = {
    'title':      '#E74C3C',
    'text':       '#3498DB',
    'aside_text': '#2ECC71',
    'table':      '#E67E22',
    'image':      '#9B59B6',
    'list':       '#1ABC9C',
    'equation':   '#F1C40F',
    'caption':    '#E91E63',
    'code':       '#795548',
    'references': '#0D47A1',
    'header':     '#FF6F00',
    'footer':     '#004D40',
    'signature':  '#607D8B',
}

# Paragraph-level block types (text body content)
PARAGRAPH_TYPES = {'text', 'title', 'list', 'aside_text', 'caption', 'references'}

print('Helpers defined.')

---
## 1. OCR-4 API Call

Single request that enables all four features simultaneously:
- `include_blocks: true` → bounding boxes + block classification
- `confidence_scores_granularity: 'page'` → per-page (regional) confidence
- `extract_header / extract_footer: true` → captures document headers/footers
- `table_format: 'markdown'` → tables rendered as markdown

In [ ]:
payload = {
    'model': MODEL_NAME,
    'document': {
        'type': 'document_url',
        'document_url': f'data:application/pdf;base64,{encode_file(NVIDIA_PDF)}',
    },
    'include_blocks': True,
    'confidence_scores_granularity': 'page',
    'extract_header': True,
    'extract_footer': True,
    'table_format': 'markdown',
}

response   = ocr_request(payload)
pages      = response['pages']
n_pages    = len(pages)

print(f'Model           : {response["model"]}')
print(f'Pages processed : {n_pages}')
print(f'Usage           : {response.get("usage", {})}')


---
## 2. Markdown Output

OCR-4 converts the entire PDF into clean markdown, preserving:
- Heading hierarchy (`#`, `##`, `###`)
- Tables (rendered per `table_format`)
- Lists, bold/italic emphasis
- Mathematical expressions (LaTeX `$…$`)
- Image references (`![…](…)`)

In [ ]:
# ── 2a. Per-page markdown statistics ────────────────────────────────────────

md_stats = []
for p in pages:
    md = p.get('markdown') or ''
    lines  = md.splitlines()
    tables = sum(1 for l in lines if l.strip().startswith('|'))
    headings = sum(1 for l in lines if l.startswith('#'))
    md_stats.append({
        'Page':          p['index'] + 1,
        'Chars':         len(md),
        'Lines':         len(lines),
        'Headings':      headings,
        'Table lines':   tables,
    })

df_md = pd.DataFrame(md_stats).set_index('Page')
display(df_md)

total_chars = df_md['Chars'].sum()
print(f'\nTotal markdown characters : {total_chars:,}')

In [ ]:
# ── 2b. Character count per page (document density) ─────────────────────────

fig, ax = plt.subplots(figsize=(14, 4))
page_nums = df_md.index.tolist()
chars     = df_md['Chars'].tolist()
bars = ax.bar(page_nums, chars, color='#3498DB', alpha=0.8, edgecolor='white')
ax.set_xlabel('Page')
ax.set_ylabel('Markdown characters')
ax.set_title('Markdown Output Length per Page — Nvidia 10-Q Form', fontweight='bold')
ax.axhline(y=np.mean(chars), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean: {int(np.mean(chars)):,} chars')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2ca. Rendered markdown preview: first 1 500 characters of page 1 ──────────

page1_md = pages[0].get('markdown', '')
display(Markdown(f'### Page 1 — Markdown Preview\n\n{page1_md[:1500]}\n\n*(truncated)*'))

In [ ]:
# ── 2d. Export full markdown to file ────────────────────────────────────────

full_md = '\n\n---\n\n'.join(
    f'<!-- Page {p["index"] + 1} -->\n{p.get("markdown", "")}'
    for p in pages
)

output_md_path = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                               'nvidia_10q_ocr4_output.md')
with open(output_md_path, 'w', encoding='utf-8') as f:
    f.write(full_md)

print(f'Full markdown saved to: {output_md_path}')
print(f'File size             : {os.path.getsize(output_md_path):,} bytes')

### 2e. Fifth option: page-6 table handling with empty cells preserved

When OCR tables contain blank cells, the main risk is that a missing value shifts the remaining columns to the left. The safest approach is to pad short rows before building a DataFrame so blanks stay in place and do not corrupt the result.

This example uses the table on page 6 of the Nvidia 10-Q and shows that empty cells are treated as missing values, not as data that moves neighboring columns.

In [ ]:
# ── 2e. Fifth table-handling option: page 6, empty cells preserved ────────


def markdown_table_rows(table_md: str) -> list[list[str]]:
    """Convert markdown table text into rows while preserving blank cells."""
    rows = []
    for line in table_md.splitlines():
        s = line.strip()
        if not s.startswith('|'):
            continue
        cells = [c.strip() for c in s.strip('|').split('|')]
        if len(cells) <= 1:
            continue
        is_separator = all(
            cell.strip().replace('-', '').replace(':', '') == ''
            for cell in cells
        )
        if is_separator:
            continue
        rows.append(cells)
    return rows


def markdown_table_to_df(table_md: str) -> pd.DataFrame:
    """Pad short rows to keep empty cells from shifting later columns."""
    rows = markdown_table_rows(table_md)
    if not rows:
        return pd.DataFrame()

    header = rows[0]
    body = []
    for row in rows[1:]:
        if len(row) < len(header):
            row = row + [''] * (len(header) - len(row))
        elif len(row) > len(header):
            row = row[:len(header)]
        body.append(row)

    df = pd.DataFrame(body, columns=header)
    return df.replace(r'^\s*$', pd.NA, regex=True)


page6 = next((p for p in pages if p.get('index') == 5), pages[5])
page6_table_md = ''
for tbl in page6.get('tables') or []:
    content = (tbl.get('content') or '').strip()
    if content:
        page6_table_md = content
        break

if page6_table_md:
    page6_df = markdown_table_to_df(page6_table_md)
    display(Markdown("### Page 6 — empty-cell-safe table parsing"))
    display(page6_df.head(10))
    print(f"\nParsed rows: {len(page6_df)}")
    print("Blank cells preserved as missing values:")
    print(page6_df.isna().sum().to_dict())
    print("\nColumns remain aligned after padding short rows:")
    display(page6_df.fillna('').astype(str).replace('nan', ''))
else:
    print('No table content was returned for page 6.')

---
## 3. Paragraph-Level Bounding Boxes

OCR-4 returns pixel coordinates for each detected block.  
**Paragraph-level blocks** include: `text`, `title`, `list`, `aside_text`, `caption`, `references`.

Each paragraph block carries:
```
top_left_x / top_left_y         (origin corner, pixels)
bottom_right_x / bottom_right_y (opposite corner, pixels)
type                             (semantic classification)
content                          (extracted text in markdown)
```

In [ ]:
# ── 3a. Collect all paragraph-level blocks across the document ───────────────

para_rows = []
for p in pages:
    dims = p.get('dimensions') or {}
    for b in (p.get('blocks') or []):
        if b['type'] in PARAGRAPH_TYPES:
            w = b['bottom_right_x'] - b['top_left_x']
            h = b['bottom_right_y'] - b['top_left_y']
            para_rows.append({
                'page':        p['index'] + 1,
                'type':        b['type'],
                'x1':          b['top_left_x'],
                'y1':          b['top_left_y'],
                'x2':          b['bottom_right_x'],
                'y2':          b['bottom_right_y'],
                'width_px':    w,
                'height_px':   h,
                'area_px2':    w * h,
                'page_width':  dims.get('width', 0),
                'page_height': dims.get('height', 0),
                'content':     b['content'],
            })

df_para = pd.DataFrame(para_rows)
print(f'Total paragraph-level blocks : {len(df_para)}')
print(f'Across pages                 : {df_para["page"].nunique()} / {n_pages}')
print()
print(df_para['type'].value_counts().to_string())

In [ ]:
# ── 3b. Paragraph bounding box table (first 20 rows) ─────────────────────────

df_display = df_para[['page', 'type', 'x1', 'y1', 'x2', 'y2',
                        'width_px', 'height_px', 'area_px2']].copy()
df_display['content_preview'] = df_para['content'].str[:60].str.replace('\n', ' ') + '…'
display(df_display.head(20))

In [ ]:
# ── 3c. Paragraph bounding box overlay on the first three pages ──────────────

def render_paragraph_boxes(page_data: dict, para_types: set, title: str = '') -> None:
    """Render paragraph-level blocks as annotated rectangles on a white canvas."""
    dims   = page_data.get('dimensions') or {}
    width  = dims.get('width',  612)
    height = dims.get('height', 792)
    blocks = [b for b in (page_data.get('blocks') or []) if b['type'] in para_types]

    fig, ax = plt.subplots(figsize=(7, 10))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_facecolor('#F8F9FA')
    ax.set_aspect('equal')

    seen_types = set()
    for idx, b in enumerate(blocks):
        x1, y1 = b['top_left_x'], b['top_left_y']
        x2, y2 = b['bottom_right_x'], b['bottom_right_y']
        btype  = b['type']
        color  = BLOCK_COLORS.get(btype, '#888888')
        w, h   = x2 - x1, y2 - y1

        ax.add_patch(patches.Rectangle(
            (x1, y1), w, h,
            linewidth=0, facecolor=color, alpha=0.14
        ))
        ax.add_patch(patches.Rectangle(
            (x1, y1), w, h,
            linewidth=1.5, edgecolor=color, facecolor='none'
        ))
        ax.text(
            x1 + 3, y1 + 13, btype,
            fontsize=5, color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.12', facecolor=color,
                      alpha=0.95, edgecolor='none'),
        )
        seen_types.add(btype)

    legend_handles = [
        patches.Patch(facecolor=BLOCK_COLORS.get(t, '#888'), label=t)
        for t in sorted(seen_types)
    ]
    ax.legend(handles=legend_handles, loc='lower right',
              fontsize=7, framealpha=0.9, title='Paragraph Types')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)')
    plt.tight_layout()
    plt.show()


# Show first 3 pages
for page_obj in pages[:3]:
    pg_num = page_obj['index'] + 1
    dims   = page_obj.get('dimensions') or {}
    n_para = sum(1 for b in (page_obj.get('blocks') or [])
                 if b['type'] in PARAGRAPH_TYPES)
    render_paragraph_boxes(
        page_obj,
        para_types=PARAGRAPH_TYPES,
        title=f'Paragraph Bounding Boxes — Page {pg_num}  '
              f'({dims.get("width", "?")}×{dims.get("height", "?")} px, '
              f'{n_para} paragraph blocks)',
    )

In [ ]:
# ── 3d. Paragraph block size distribution ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Width distribution
axes[0].hist(df_para['width_px'], bins=30, color='#3498DB', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Block count')
axes[0].set_title('Paragraph Block Width', fontweight='bold')

# Height distribution
axes[1].hist(df_para['height_px'], bins=30, color='#E74C3C', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Height (px)')
axes[1].set_ylabel('Block count')
axes[1].set_title('Paragraph Block Height', fontweight='bold')

# Area scatter by type
for btype in df_para['type'].unique():
    subset = df_para[df_para['type'] == btype]
    axes[2].scatter(subset['width_px'], subset['height_px'],
                    label=btype, color=BLOCK_COLORS.get(btype, '#888'),
                    alpha=0.6, s=30)
axes[2].set_xlabel('Width (px)')
axes[2].set_ylabel('Height (px)')
axes[2].set_title('Width × Height by Block Type', fontweight='bold')
axes[2].legend(fontsize=7)

plt.suptitle('Paragraph-Level Bounding Box Geometry — Nvidia 10-Q',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3e. Paragraph blocks per page ────────────────────────────────────────────

para_per_page = df_para.groupby('page').size().reindex(range(1, n_pages + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(para_per_page.index, para_per_page.values,
       color='#2ECC71', alpha=0.85, edgecolor='white')
ax.set_xlabel('Page')
ax.set_ylabel('Paragraph block count')
ax.set_title('Paragraph-Level Blocks per Page — Nvidia 10-Q', fontweight='bold')
ax.axhline(y=para_per_page.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean: {para_per_page.mean():.1f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Regional-Level Confidence

With `confidence_scores_granularity: 'page'`, OCR-4 returns per-page (regional) confidence metrics:

| Field | Description |
|-------|-------------|
| `average_page_confidence_score` | Mean confidence across all tokens on the page |
| `minimum_page_confidence_score` | Lowest single-token confidence — flags problematic regions |

Scores range **0 → 1**. Pages below ~0.85 may contain low-resolution regions, unusual fonts, or complex layouts.

In [ ]:
# ── 4a. Extract per-page confidence scores ────────────────────────────────────

conf_rows = []
for p in pages:
    cs = p.get('confidence_scores') or {}
    avg = cs.get('average_page_confidence_score')
    mn  = cs.get('minimum_page_confidence_score')
    conf_rows.append({
        'page':    p['index'] + 1,
        'avg':     avg,
        'minimum': mn,
        'range':   (avg - mn) if (avg is not None and mn is not None) else None,
    })

df_conf = pd.DataFrame(conf_rows).set_index('page')

print('Regional (page-level) confidence scores:')
print()
display(df_conf.style.format({
    'avg':     '{:.4f}',
    'minimum': '{:.4f}',
    'range':   '{:.4f}',
}).background_gradient(subset=['avg'], cmap='RdYlGn', vmin=0.8, vmax=1.0))

In [ ]:
# ── 4b. Per-page confidence bar chart ────────────────────────────────────────

valid = df_conf.dropna(subset=['avg'])

fig, ax = plt.subplots(figsize=(14, 5))

bar_colors = ['#E74C3C' if v < 0.85 else '#2ECC71' for v in valid['avg']]
bars = ax.bar(valid.index, valid['avg'], color=bar_colors, alpha=0.85,
               edgecolor='white', label='Avg confidence')

# Plot minimum as scatter overlay
ax.scatter(valid.index, valid['minimum'], color='#E67E22', zorder=5,
           s=40, label='Min confidence')

ax.axhline(y=0.90, color='gray',    linestyle='--', alpha=0.5, label='0.90 threshold')
ax.axhline(y=0.85, color='orange',  linestyle=':',  alpha=0.6, label='0.85 threshold')
ax.axhline(y=valid['avg'].mean(), color='#2980B9', linestyle='-', linewidth=1.5,
           label=f'Doc mean {valid["avg"].mean():.4f}')

ax.set_ylim(0, 1.08)
ax.set_xlabel('Page (region)')
ax.set_ylabel('Confidence score')
ax.set_title('Regional (Page-Level) Confidence — Nvidia 10-Q Form', fontweight='bold')
ax.legend(fontsize=9, loc='lower left')

for bar, val in zip(bars, valid['avg']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=6.5)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4c. Confidence heatmap strip (avg vs minimum per page) ───────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 3.5), sharex=True)

for ax, col, label, cmap in zip(
    axes,
    ['avg', 'minimum'],
    ['Average Confidence', 'Minimum Confidence'],
    ['RdYlGn', 'RdYlGn'],
):
    vals = valid[col].values.reshape(1, -1)
    im = ax.imshow(vals, aspect='auto', cmap=cmap, vmin=0.75, vmax=1.0)
    ax.set_yticks([0])
    ax.set_yticklabels([label], fontsize=9)
    ax.set_xticks(range(len(valid)))
    ax.set_xticklabels([f'p{i}' for i in valid.index], fontsize=7)
    plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.015, pad=0.01)
    for j, v in enumerate(vals[0]):
        ax.text(j, 0, f'{v:.3f}', ha='center', va='center', fontsize=6,
                color='black' if v > 0.85 else 'white')

plt.suptitle('Confidence Heatmap — Nvidia 10-Q (green = high, red = low)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4d. Confidence summary statistics ────────────────────────────────────────

avg_scores = valid['avg'].dropna()
min_scores = valid['minimum'].dropna()

summary = pd.DataFrame({
    'Metric':         ['Document mean', 'Document min', 'Document max',
                       'Pages ≥ 0.90',  'Pages ≥ 0.85', 'Pages < 0.85'],
    'Avg confidence': [
        f'{avg_scores.mean():.4f}',
        f'{avg_scores.min():.4f}  (page {avg_scores.idxmin()})',
        f'{avg_scores.max():.4f}  (page {avg_scores.idxmax()})',
        f'{(avg_scores >= 0.90).sum()} / {len(avg_scores)}',
        f'{(avg_scores >= 0.85).sum()} / {len(avg_scores)}',
        f'{(avg_scores < 0.85).sum()} / {len(avg_scores)}',
    ],
    'Min confidence': [
        f'{min_scores.mean():.4f}',
        f'{min_scores.min():.4f}  (page {min_scores.idxmin()})',
        f'{min_scores.max():.4f}  (page {min_scores.idxmax()})',
        f'{(min_scores >= 0.90).sum()} / {len(min_scores)}',
        f'{(min_scores >= 0.85).sum()} / {len(min_scores)}',
        f'{(min_scores < 0.85).sum()} / {len(min_scores)}',
    ],
}).set_index('Metric')

display(summary)

---
## 5. Bounding Box Classification

Every block is classified into one of **13 semantic types**:

| Category | Types |
|----------|-------|
| **Structure** | `title`, `header`, `footer` |
| **Paragraph** | `text`, `list`, `aside_text`, `caption`, `references` |
| **Specialised** | `table`, `image`, `equation`, `code`, `signature` |

The classification enables downstream tasks like:
- Table extraction (filter `table` blocks)
- Figure captioning (pair `image` with adjacent `caption` blocks)
- Header/footer stripping for clean body text

In [ ]:
# ── 5a. Collect ALL blocks (all types) across every page ─────────────────────

all_rows = []
for p in pages:
    dims = p.get('dimensions') or {}
    for b in (p.get('blocks') or []):
        w = b['bottom_right_x'] - b['top_left_x']
        h = b['bottom_right_y'] - b['top_left_y']
        all_rows.append({
            'page':        p['index'] + 1,
            'type':        b['type'],
            'x1':          b['top_left_x'],
            'y1':          b['top_left_y'],
            'x2':          b['bottom_right_x'],
            'y2':          b['bottom_right_y'],
            'width_px':    w,
            'height_px':   h,
            'area_px2':    w * h,
            'content_len': len(b['content']),
        })

df_all = pd.DataFrame(all_rows)
print(f'Total blocks across {n_pages} pages: {len(df_all)}')
print()
print(df_all['type'].value_counts().to_string())

In [ ]:
# ── 5b. Block-type distribution: bar + pie ────────────────────────────────────

type_counts = df_all['type'].value_counts()
colors      = [BLOCK_COLORS.get(t, '#888888') for t in type_counts.index]

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = ax_bar.barh(type_counts.index, type_counts.values,
                    color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, type_counts.values):
    ax_bar.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)
ax_bar.set_xlabel('Block count')
ax_bar.set_title('Block Counts by Type\n(Nvidia 10-Q, all pages)', fontweight='bold')

# Pie chart
threshold = 0.02 * type_counts.sum()
major     = type_counts[type_counts >= threshold].copy()
minor_sum = int(type_counts[type_counts < threshold].sum())
if minor_sum > 0:
    major = pd.concat([major, pd.Series([minor_sum], index=['other'])])
pie_labels = list(major.index)
pie_colors = [BLOCK_COLORS.get(lbl, '#AAAAAA') for lbl in pie_labels]
ax_pie.pie(major.values, labels=pie_labels, colors=pie_colors,
           autopct='%1.1f%%', startangle=140, pctdistance=0.82)
ax_pie.set_title('Block Type Proportions', fontweight='bold')

plt.suptitle('OCR-4 Block Classification — Nvidia 10-Q Form',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5c. Block-type heat-map: type × page ─────────────────────────────────────

pivot = (
    df_all.groupby(['type', 'page'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, n_pages + 1), fill_value=0)
)

fig, ax = plt.subplots(figsize=(max(10, n_pages * 0.7), len(pivot) * 0.7 + 1.5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(n_pages))
ax.set_xticklabels([f'p{i}' for i in range(1, n_pages + 1)], fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_xlabel('Page')
ax.set_ylabel('Block Type')
ax.set_title('Blocks per Page by Type — Nvidia 10-Q Form', fontsize=12, fontweight='bold')

for i in range(len(pivot.index)):
    for j in range(n_pages):
        val = pivot.values[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=7, color='black')

plt.colorbar(im, ax=ax, label='Block count')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5d. Full block classification overlay on first three pages ───────────────

def render_all_blocks(page_data: dict, title: str = '') -> None:
    """Render ALL block types (including tables, images, headers) on one canvas."""
    dims   = page_data.get('dimensions') or {}
    width  = dims.get('width',  612)
    height = dims.get('height', 792)
    blocks = page_data.get('blocks') or []

    fig, ax = plt.subplots(figsize=(7, 10))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_facecolor('#F8F9FA')
    ax.set_aspect('equal')

    seen_types = set()
    for b in blocks:
        x1, y1 = b['top_left_x'], b['top_left_y']
        x2, y2 = b['bottom_right_x'], b['bottom_right_y']
        btype  = b['type']
        color  = BLOCK_COLORS.get(btype, '#888888')
        w, h   = x2 - x1, y2 - y1

        ax.add_patch(patches.FancyBboxPatch(
            (x1, y1), w, h,
            boxstyle='round,pad=2',
            linewidth=0, facecolor=color, alpha=0.18,
        ))
        ax.add_patch(patches.Rectangle(
            (x1, y1), w, h,
            linewidth=1.5, edgecolor=color, facecolor='none'
        ))
        ax.text(
            x1 + 3, y1 + 13, btype,
            fontsize=5, color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.12', facecolor=color,
                      alpha=0.95, edgecolor='none'),
        )
        seen_types.add(btype)

    legend_handles = [
        patches.Patch(facecolor=BLOCK_COLORS.get(t, '#888'), label=t)
        for t in sorted(seen_types)
    ]
    ax.legend(handles=legend_handles, loc='lower right',
              fontsize=7, framealpha=0.9, title='Block Types')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('x (pixels)')
    ax.set_ylabel('y (pixels)')
    plt.tight_layout()
    plt.show()


for page_obj in pages[:3]:
    pg_num = page_obj['index'] + 1
    n_blk  = len(page_obj.get('blocks') or [])
    render_all_blocks(
        page_obj,
        title=f'Block Classification — Page {pg_num}  ({n_blk} blocks)',
    )

In [ ]:
# ── 5e. One example per detected block type ──────────────────────────────────

print('Block Classification Examples — Nvidia 10-Q')
print('=' * 72)
for btype in df_all['type'].unique():
    row     = df_all[df_all['type'] == btype].iloc[0]
    # look up raw content from pages
    page_blocks = pages[row['page'] - 1].get('blocks') or []
    match = next(
        (b for b in page_blocks
         if b['type'] == btype
         and b['top_left_x'] == row['x1']
         and b['top_left_y'] == row['y1']),
        None
    )
    content = match['content'][:120].replace('\n', ' ') if match else '(n/a)'
    count   = (df_all['type'] == btype).sum()
    print(f'\n  TYPE : {btype.upper():12s}  ({count} across doc)')
    print(f'  PAGE : {row["page"]}')
    print(f'  BBOX : ({row["x1"]}, {row["y1"]}) → ({row["x2"]}, {row["y2"]})  '
          f'[{row["width_px"]}×{row["height_px"]}px]')
    print(f'  TEXT : {content}{"…" if len(content) == 120 else ""}')
    print('-' * 72)

In [ ]:
# ── 5f. Average bounding-box area per block type ─────────────────────────────

area_stats = (
    df_all.groupby('type')['area_px2']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .sort_values('mean', ascending=False)
    .rename(columns={'mean': 'avg_area', 'median': 'med_area',
                     'min': 'min_area', 'max': 'max_area', 'count': 'n'})
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = [BLOCK_COLORS.get(t, '#888888') for t in area_stats.index]
x_pos  = np.arange(len(area_stats))
ax.bar(x_pos, area_stats['avg_area'] / 1_000, color=colors, alpha=0.8)
ax.errorbar(
    x=x_pos,
    y=area_stats['avg_area'].to_numpy() / 1_000,
    yerr=[
        (area_stats['avg_area'] - area_stats['min_area']).to_numpy() / 1_000,
        (area_stats['max_area'] - area_stats['avg_area']).to_numpy() / 1_000,
    ],
    fmt='none', color='gray', capsize=4, linewidth=1,
)
ax.set_xticks(x_pos)
ax.set_xticklabels(area_stats.index, rotation=20, ha='right')
ax.set_ylabel('Average area (×1,000 px²)')
ax.set_xlabel('Block type')
ax.set_title('Mean Bounding-Box Area by Block Type — Nvidia 10-Q\n(error bars = min/max)',
             fontweight='bold')
plt.tight_layout()
plt.show()

display(area_stats.style.format({
    'avg_area': '{:,.0f}',
    'med_area': '{:,.0f}',
    'min_area': '{:,.0f}',
    'max_area': '{:,.0f}',
}))

---
## 6. Summary

In [ ]:
# ── 6. Final summary dashboard ────────────────────────────────────────────────

avg_conf    = valid['avg'].mean() if len(valid) else float('nan')
min_conf    = valid['minimum'].min() if len(valid) else float('nan')
n_para_blk  = len(df_para)
n_all_blk   = len(df_all)
types_found = sorted(df_all['type'].unique())

summary_md = f"""
### OCR-4 Results — Nvidia 10-Q Form

| Metric | Value |
|--------|-------|
| Pages processed | {n_pages} |
| Total markdown chars | {df_md['Chars'].sum():,} |
| Paragraph-level blocks | {n_para_blk} ({', '.join(sorted(PARAGRAPH_TYPES & set(types_found)))}) |
| All block types detected | {len(types_found)}: {', '.join(f'`{t}`' for t in types_found)} |
| Total blocks (all types) | {n_all_blk} |
| Doc avg confidence | {avg_conf:.4f} |
| Doc min confidence | {min_conf:.4f} |
| Pages ≥ 0.90 confidence | {(valid['avg'] >= 0.90).sum()} / {len(valid)} |
"""

display(Markdown(summary_md))